In [1]:
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
import numpy as np
import pandas as pd

In [2]:
df = pd.read_parquet("data/processed/model_table.parquet")
y = df["default_flag"]
x = df.drop(columns=["id", "default_flag", "issue_year", "issue_date", "funded_amnt", "annual_inc"])
x = pd.get_dummies(x, columns=["home_ownership", "verification_status", "purpose"], drop_first=True, dtype=int)

print(f"Shape of x: {x.shape}")
print(f"Mean of y: {y.mean()}")

Shape of x: (545779, 80)
Mean of y: 0.14068331687367963


In [3]:
train_mask = df["issue_year"] <= 2014
x_train = x.loc[train_mask]
y_train = y.loc[train_mask]
x_test = x.loc[~train_mask]
y_test = y.loc[~train_mask]

print(f"Rows in train set: {y_train.shape[0]}, Rows in test set: {y_test.shape[0]}")
print(f"Mean of y_train: {y_train.mean()}, Mean of y_test: {y_test.mean()}")

Rows in train set: 262992, Rows in test set: 282787
Mean of y_train: 0.1319165601995498, Mean of y_test: 0.14883640337073487


In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression 
from sklearn.metrics import roc_auc_score 

scaler = StandardScaler()
x_train_s = scaler.fit_transform(x_train)
x_test_s = scaler.transform(x_test)

model = LogisticRegression(max_iter=1000)
model.fit(x_train_s, y_train)

pd_test = model.predict_proba(x_test_s)[:, 1]

print(f"roc_auc_score: {roc_auc_score(y_test, pd_test)}")
print(f"roc_auc_score (train): {roc_auc_score(y_train, model.predict_proba(x_train_s)[:, 1])}")
print(f"mean of pd_test: {pd_test.mean()}")

roc_auc_score: 0.683909518394715
roc_auc_score (train): 0.6773873853202709
mean of pd_test: 0.13617472692738952


In [5]:
import duckdb
conn = duckdb.connect('data/loans.duckdb', read_only=True)

grades = conn.sql("SELECT id, sub_grade FROM loans").df()
grades_merged = df.loc[~train_mask].merge(grades, on="id", how="left")
print(f"Rows in merged: {grades_merged.shape[0]}")
print(f"Example sub_grade: {grades_merged['sub_grade'].iloc[100]}")

grade_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6}
grades_merged["sub_grade_num"] = grades_merged["sub_grade"].apply(lambda s: grade_map.get(s[0], 0) * 5 + int(s[1]))
print(f"ROC AUC Score: {roc_auc_score(grades_merged['default_flag'], grades_merged['sub_grade_num'])}")
print(f"5 Example sub_grade_num: {grades_merged['sub_grade_num'].iloc[100]}, {grades_merged['sub_grade_num'].iloc[101]}, {grades_merged['sub_grade_num'].iloc[102]}, {grades_merged['sub_grade_num'].iloc[103]}, {grades_merged['sub_grade_num'].iloc[104]}")

Rows in merged: 282787
Example sub_grade: B2
ROC AUC Score: 0.6786417383122245
5 Example sub_grade_num: 7, 4, 18, 8, 10


In [6]:
from sklearn.metrics import roc_curve
fpr, tpr, thresholds = roc_curve(y_test, pd_test)
ks_lr = (tpr-fpr).max()
print(f"KS Statistic (Logistic Regression): {ks_lr}")

fpr, tpr, thresholds = roc_curve(grades_merged['default_flag'], grades_merged['sub_grade_num'])
ks_grade = (tpr-fpr).max()
print(f"KS Statistic (Sub Grade Benchmark): {ks_grade}")

df2 = pd.DataFrame({"y_test": y_test, "pd_test": pd_test})
decile = pd.qcut(pd_test, 10, labels=False)
calib_table = df2.groupby(decile).agg({"y_test": ["count", "mean"], "pd_test": ["mean"]}).sort_index(ascending=False)
print(calib_table)

KS Statistic (Logistic Regression): 0.2672289009347664
KS Statistic (Sub Grade Benchmark): 0.26194575341328713
  y_test             pd_test
   count      mean      mean
9  28279  0.319460  0.308101
8  28279  0.237986  0.218123
7  28278  0.201358  0.179522
6  28279  0.172425  0.152488
5  28278  0.147182  0.130640
4  28279  0.124403  0.111545
3  28279  0.100746  0.093895
2  28278  0.086074  0.076668
1  28279  0.062838  0.058137
0  28279  0.035892  0.032625


In [7]:
import statsmodels.api as sm
x_train_df = pd.DataFrame(x_train_s, columns=x_train.columns)
x_train_df = sm.add_constant(x_train_df)
logit = sm.Logit(y_train.values, x_train_df).fit()

params_table = pd.DataFrame({"coefficients": logit.params, "p_values": logit.pvalues}).sort_values(by="coefficients", key=abs, ascending=False)
print(f"Top 15 coefficients:\n{params_table.head(15)}")

Optimization terminated successfully.
         Current function value: 0.366999
         Iterations 7
Top 15 coefficients:
                             coefficients      p_values
const                           -2.049339  0.000000e+00
total_bal_ex_mort                0.311123  4.968643e-26
total_il_high_credit_limit      -0.280214  1.217241e-24
open_acc                        -0.258196  4.773171e-02
num_sats                         0.213584  1.013944e-01
mths_since_last_delinq          -0.211501  6.520951e-05
total_rev_hi_lim                -0.207985  1.290286e-04
fico                            -0.203430  1.000149e-82
ever_mths_since_last_delinq     -0.196391  9.516177e-05
total_bc_limit                  -0.178524  2.842516e-15
mo_sin_old_rev_tl_op            -0.170198  1.168215e-26
acc_open_past_24mths             0.165482  7.105378e-75
dti                              0.151901  1.857124e-77
percent_bc_gt_75                 0.141980  1.175375e-39
credit_history_months            0.13

In [8]:
corr = x_train.corr().abs()
mask = np.triu(np.ones(corr.shape), k=1).astype(bool)
pairs = corr.where(mask).stack()

pairs.dropna()
pairs.drop(pairs[pairs < 0.8].index)
pairs.sort_values(ascending=False, inplace=True)

print(f"Number of highly correlated pairs: {len(pairs)}")
print(f"All pairs in order of correlation:\n{pairs}")

Number of highly correlated pairs: 6400
All pairs in order of correlation:
open_acc                        num_sats                               0.998712
num_actv_rev_tl                 num_rev_tl_bal_gt_0                    0.991560
mths_since_last_major_derog     ever_mths_since_last_major_derog       0.986451
mths_since_recent_bc_dlq        ever_mths_since_recent_bc_dlq          0.985391
mths_since_recent_revol_delinq  ever_mths_since_recent_revol_delinq    0.982554
                                                                         ...   
purpose_vacation                purpose_medical                             NaN
                                purpose_moving                              NaN
                                purpose_other                               NaN
                                purpose_small_business                      NaN
                                purpose_vacation                            NaN
Length: 6400, dtype: float64


**Feature pruning (multicollinearity)**
Pairs with absolute correlation > 0.8, computed on training data (2013–14) only: 24 pairs found.
VIF (variance inflation factor) on a 50,000-row sample of scaled training data: dropped or justified non-dummy features with VIF > 10. VIF for purpose_ dummies was ignored (encoding artifact of the small reference category, car).

**Features dropped (23):**
*Family 1 – Account counts*: num_sats, num_rev_tl_bal_gt_0, total_acc, num_op_rev_tl, num_rev_accts
*Family 2 – Balances & limits*: tot_hi_cred_lim, total_il_high_credit_limit, total_bc_limit, revol_bal, total_rev_hi_lim
*Family 3 – Utilization*: bc_util
*Family 4 – Delinquency*: ever_mths_since_last_major_derog, ever_mths_since_recent_bc_dlq, ever_mths_since_recent_revol_delinq, ever_mths_since_last_delinq, ever_mths_since_last_record, pct_tl_nvr_dlq, pub_rec, mths_since_last_record, mths_since_recent_revol_delinq, num_tl_90g_dpd_24m
*Family 5 – Credit history age*: mo_sin_old_rev_tl_op, mo_sin_rcnt_rev_tl_op

**Reasoning:**
1) I chose open_acc over num_sats because open accounts is a more useful metric than satisfactory accounts to determine total accounts open, and at 0.999 correlation, num_sats adds no information. I dropped total_acc because it is roughly the sum of revolving and installment accounts, a three-way duplicate that pairwise correlation missed. I dropped num_op_rev_tl and num_rev_accts because their VIF was above 10 and open_acc already captures how many accounts are open.
2) I chose total balance over total limit because limits are approval-based numbers set by lenders, while balances are actually in the hands of the borrower to change. I dropped total_bc_limit because it caused bc_open_to_buy to take a backwards sign. I dropped revol_bal and total_rev_hi_lim because revol_util is built from them (balance ÷ limit), so keeping all three was redundant.
3) I chose revol_util over bc_util because while bank cards are a majority of revolving credit, they are not representative of all credit used, and revol_util is the standard utilization figure on a credit report.
4) I chose months since over the ever flags because the months-since metric shows changes in creditworthiness over time, not just a single poor event. I kept ever_mths_since_recent_inq because it had no correlated twin. I dropped pct_tl_nvr_dlq because it took a backwards sign due to overlap with the delinquency features. I dropped pub_rec because it is roughly the sum of pub_rec_bankruptcies and tax_liens. I dropped mths_since_last_record, mths_since_recent_revol_delinq, and num_tl_90g_dpd_24m because they took small backwards signs and would produce nonsensical reason codes.
5) I chose my credit_history_months over the bureau's mo_sin_old_rev_tl_op because it counts from the first account of any type, not just revolving accounts, matching FICO's "length of credit history" factor. I dropped mo_sin_rcnt_rev_tl_op because it took a backwards sign while overlapping with mo_sin_rcnt_tl.

**Documented exceptions (kept):**
- pub_rec_bankruptcies (−0.021): slightly protective after controlling for other factors; possibly because borrowers with a past bankruptcy were screened more heavily. Cannot produce a misleading reason code since reason codes only use factors that increase PD.
- verification_status_Verified (+0.027): verified income appears riskier because Lending Club requested verification more often from applicants who already looked risky.

**Result:** AUC 0.684 → 0.679 with 80 → 57 features; no non-dummy VIF > 10; all significant coefficients have sensible signs except the two documented exceptions.

In [9]:
prune_cols = [
    "num_sats", "num_rev_tl_bal_gt_0", "total_acc", "tot_hi_cred_lim",
    "total_il_high_credit_limit", "total_bc_limit", "bc_util",
    "ever_mths_since_last_major_derog", "ever_mths_since_recent_bc_dlq",
    "ever_mths_since_recent_revol_delinq", "ever_mths_since_last_delinq",
    "ever_mths_since_last_record", "pct_tl_nvr_dlq", "mo_sin_old_rev_tl_op", 
    "revol_bal", "total_rev_hi_lim", "pub_rec", "num_op_rev_tl", "num_rev_accts",
    "mo_sin_rcnt_rev_tl_op", "mths_since_last_record", "mths_since_recent_revol_delinq", "num_tl_90g_dpd_24m"
]
x_train_p = x_train.drop(columns=prune_cols)
x_test_p = x_test.drop(columns=prune_cols)

sc = StandardScaler()
x_train_p_s = sc.fit_transform(x_train_p)
x_test_p_s = sc.transform(x_test_p)

model_p = LogisticRegression(max_iter=1000)
model_p.fit(x_train_p_s, y_train)
pd_p_test = model_p.predict_proba(x_test_p_s)[:, 1]

print(f"roc_auc_score: {roc_auc_score(y_test, pd_p_test)}")

x_train_p_df = pd.DataFrame(x_train_p_s, columns=x_train_p.columns)
x_train_p_df = sm.add_constant(x_train_p_df)
logit_p = sm.Logit(y_train.values, x_train_p_df).fit()

params_p_table = pd.DataFrame({"coefficients": logit_p.params, "p_values": logit_p.pvalues}).sort_values(by="coefficients", key=abs, ascending=False)
print(f"Top 15 coefficients:\n{params_p_table.head(16)}")

roc_auc_score: 0.6790787997036727
Optimization terminated successfully.
         Current function value: 0.368232
         Iterations 7
Top 15 coefficients:
                        coefficients      p_values
const                      -2.039997  0.000000e+00
fico                       -0.210987  5.222454e-95
acc_open_past_24mths        0.174079  4.043632e-85
loan_to_income              0.144149  2.987951e-23
log_annual_inc             -0.137692  4.312199e-18
bc_open_to_buy             -0.113374  7.799753e-22
dti                         0.110210  1.896620e-51
home_ownership_RENT         0.098428  3.247518e-34
percent_bc_gt_75            0.097101  7.374377e-31
inq_last_6mths              0.088960  2.534409e-32
purpose_small_business      0.084853  1.081376e-24
num_tl_op_past_12m          0.083003  3.905915e-22
emp_length_missing          0.074339  4.712722e-39
num_il_tl                  -0.073993  1.078241e-17
mths_since_recent_inq      -0.069048  5.881769e-10
purpose_credit_card        

In [10]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
X_vif = pd.DataFrame(x_train_p_s, columns=x_train_p.columns).sample(50000, random_state=0)
X_vif = sm.add_constant(X_vif)

vif = pd.Series(
    {
        X_vif.columns[i]: variance_inflation_factor(X_vif.values, i)
        for i in range(1, X_vif.shape[1])
    },
    name="VIF",
).sort_values(ascending=False)

print(f"Top 20 VIF values:\n{vif.head(20)}")
print(f"Features with VIF > 10: {(vif > 10).sum()}")
print(f"Features with VIF > 5: {(vif > 5).sum()}")

Top 20 VIF values:
purpose_debt_consolidation    27.265179
purpose_credit_card           21.539699
tot_cur_bal                    7.966887
loan_amnt                      7.523021
log_annual_inc                 6.920098
purpose_home_improvement       6.390187
purpose_other                  6.368388
loan_to_income                 6.292112
num_actv_bc_tl                 6.174410
avg_cur_bal                    6.044409
num_bc_sats                    6.013488
acc_now_delinq                 4.890232
num_actv_rev_tl                4.320833
open_acc                       4.110350
num_tl_30dpd                   4.061591
mths_since_recent_inq          3.355417
purpose_major_purchase         2.969026
revol_util                     2.738160
acc_open_past_24mths           2.563046
total_bal_ex_mort              2.451772
Name: VIF, dtype: float64
Features with VIF > 10: 2
Features with VIF > 5: 11


In [11]:
import lightgbm as lgb

lgbm = lgb.LGBMClassifier(
    n_estimators=400,
    learning_rate=.05,
    num_leaves=31,
    min_child_samples=200,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    random_state=0, 
    verbose=-1
)

lgbm.fit(x_train, y_train)

pd_lgbm_train = lgbm.predict_proba(x_train)[:, 1]
pd_lgbm_test = lgbm.predict_proba(x_test)[:, 1]

train_auc_lgbm = roc_auc_score(y_train, pd_lgbm_train)
test_auc_lgbm = roc_auc_score(y_test, pd_lgbm_test)
fpr_lgbm, tpr_lgbm, thresholds_lgbm = roc_curve(y_test, pd_lgbm_test)
ks_lgbm = (tpr_lgbm - fpr_lgbm).max()
actual_test_default_rate = y_test.mean()

print(f"LightGBM train AUC: {train_auc_lgbm:.4f}")
print(f"LightGBM test AUC: {test_auc_lgbm:.4f}")
print(f"LightGBM test KS (ks_lgbm): {ks_lgbm:.4f}")
print(f"Mean predicted test PD: {pd_lgbm_test.mean():.4f}")
print(f"Actual test default rate: {actual_test_default_rate:.4f}")

LightGBM train AUC: 0.7535
LightGBM test AUC: 0.6960
LightGBM test KS (ks_lgbm): 0.2843
Mean predicted test PD: 0.1372
Actual test default rate: 0.1488


In [12]:
import json
from pathlib import Path

train_auc_lr_p = roc_auc_score(y_train, model_p.predict_proba(x_train_p_s)[:, 1])
test_auc_lr_full = roc_auc_score(y_test, pd_test)
test_auc_lr_p = roc_auc_score(y_test, pd_p_test)
test_auc_lgbm = roc_auc_score(y_test, pd_lgbm_test)
train_auc_lgbm = roc_auc_score(y_train, pd_lgbm_train)

fpr_lr_p, tpr_lr_p, _ = roc_curve(y_test, pd_p_test)
ks_lr_p = (tpr_lr_p - fpr_lr_p).max()
fpr_lr_full, tpr_lr_full, _ = roc_curve(y_test, pd_test)
ks_lr_full = (tpr_lr_full - fpr_lr_full).max()
fpr_grade, tpr_grade, _ = roc_curve(grades_merged["default_flag"], grades_merged["sub_grade_num"])
ks_grade = (tpr_grade - fpr_grade).max()

metrics = {
    "data": {
        "train_rows": int(len(y_train)),
        "test_rows": int(len(y_test)),
        "actual_test_default_rate": round(float(y_test.mean()), 4),
    },
    "lr_80": {
        "test_auc": round(float(test_auc_lr_full), 4),
        "ks": round(float(ks_lr_full), 4),
    },
    "lr_57": {
        "test_auc": round(float(test_auc_lr_p), 4),
        "train_auc": round(float(train_auc_lr_p), 4),
        "ks": round(float(ks_lr_p), 4),
        "gini": round(float(2 * test_auc_lr_p - 1), 4),
        "mean_predicted_pd": round(float(pd_p_test.mean()), 4),
    },
    "lightgbm_80": {
        "test_auc": round(float(test_auc_lgbm), 4),
        "train_auc": round(float(train_auc_lgbm), 4),
        "ks": round(float(ks_lgbm), 4),
        "mean_predicted_pd": round(float(pd_lgbm_test.mean()), 4),
    },
    "lc_sub_grade_benchmark": {
        "auc": round(float(roc_auc_score(grades_merged["default_flag"], grades_merged["sub_grade_num"])), 4),
        "ks": round(float(ks_grade), 4),
    },
    "chosen_model": "logistic_regression_pruned_57",
    "reason": f"The pruned 57 feature regression is chosen over the LightGBM (+{test_auc_lgbm - test_auc_lr_p:.4f} AUC) because it does not show signs of overfitting, its coefficients provide adverse-action reasons, and performs on par with the sub grade form Lending Club",
}

Path("outputs").mkdir(parents=True, exist_ok=True)
with open("outputs/metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

scored_test = df.loc[~train_mask, ["id", "default_flag", "funded_amnt"]].copy()
scored_test["pd_lr"] = pd_p_test
scored_test["pd_lgbm"] = pd_lgbm_test
Path("data/processed").mkdir(parents=True, exist_ok=True)
scored_test.to_parquet("data/processed/scored_test.parquet", index=False)

print(json.dumps(metrics, indent=2))
print(f"Saved {len(scored_test):,} scored test loans to data/processed/scored_test.parquet")

{
  "data": {
    "train_rows": 262992,
    "test_rows": 282787,
    "actual_test_default_rate": 0.1488
  },
  "lr_80": {
    "test_auc": 0.6839,
    "ks": 0.2672
  },
  "lr_57": {
    "test_auc": 0.6791,
    "train_auc": 0.6727,
    "ks": 0.2594,
    "gini": 0.3582,
    "mean_predicted_pd": 0.138
  },
  "lightgbm_80": {
    "test_auc": 0.696,
    "train_auc": 0.7535,
    "ks": 0.2843,
    "mean_predicted_pd": 0.1372
  },
  "lc_sub_grade_benchmark": {
    "auc": 0.6786,
    "ks": 0.2619
  },
  "chosen_model": "logistic_regression_pruned_57",
  "reason": "The pruned 57 feature regression is chosen over the LightGBM (+0.0169 AUC) because it does not show signs of overfitting, its coefficients provide adverse-action reasons, and performs on par with the sub grade form Lending Club"
}
Saved 282,787 scored test loans to data/processed/scored_test.parquet
